# Lecture 10 - Context and verification

**Lecture Topic**: The agentic workflow, managing what the agent sees, and giving it ways to check its own work

## Announcements

- Thanks for your project proposals! Feedback will come back on Gradescope.
- The midterm is next Monday, Oct 26, in class. It's closed-AI and covers Modules A and B.
- ClassBuzz poll for today: `RFKS`

## Recap

In the last lecture, we covered the following topics:
 - Tool calling: the model only asks, and the harness runs the tool
 - The agent loop, and how much a small harness change can fix
 - The anatomy of a harness
 - A real Claude Code session, step by step, and what it cost

Today, we will cover the following topics:
 - The agentic workflow: requirements, spec, plan, implement, verify
 - The same task from a vague request and from a precise spec
 - Writing a spec
 - Context management: what goes into the context, and what should stay out
 - Verification: the same task with and without a test suite

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

taxis = pd.read_csv('../data/2019_Yellow_Taxi_Trip_Data.csv')

def load_session(name):
    return [json.loads(line) for line in open(f'../data/claude_code_session_{name}.jsonl')]

# The agentic workflow

So far we've seen *how* an agent works. Now let's look at how to work *with* one.

In Lecture 1 we asked a vague question ("which zones tip the best?") and got a plausible, wrong answer. And in the session we traced last time, the agent made two judgment calls of its own that changed the result.

Neither was really the agent's fault. When a request leaves decisions open, the agent has to make them for you, and it will usually do so quietly and reasonably. The way to keep control is to make those decisions yourself, *before* any code gets written.

## Requirements → spec → plan → implement → verify

| Step | The question | Who does the work |
|---|---|---|
| **Requirements** | What does the user actually need, and why? | You, talking to people |
| **Spec** | What exactly should be built: definitions, rules, outputs, what "done" means | You (the agent can help draft and poke holes in it) |
| **Plan** | How will it be built? Which files, which steps, which libraries? | The agent proposes, you review |
| **Implement** | Write the code | The agent |
| **Verify** | Does it meet the spec? | The agent runs the checks, you review the evidence |

Note, this is a loop rather than a straight line: verifying often turns up questions that send you back to the spec.

The middle steps are now cheap and fast, so the steps at either end are where your time goes. A good spec is also exactly what you need to *verify* the result, since it tells you what to check.

## Case study: the same task, two ways

I gave Claude Code (Opus 5.5) the same taxi data and the same `CLAUDE.md` (where Python lives, and that full monthly files are ~7 million rows) in two separate folders, with the same permissions as before.

**Vague:**
> *"Can you write a script that cleans up the taxi data so we can use it for analysis?"*

**Precise:**
> *"Please implement SPEC.md."*

### The spec

```markdown
# Spec: clean_taxis.py

## Goal and user
A cleaned copy of the taxi trips for the team's tip and speed analyses. It will be re-run on each new monthly file.

## Input
`data/taxi_2019.csv` (10,000-row sample). Full monthly files are about 7 million rows with the same columns.

## Output
- `data/taxi_clean.csv`: the kept trips, all original columns, plus `duration_min` and `speed_mph`.
- A printed summary: rows in, rows out, and the number of rows dropped by each rule.
```

```markdown
## Rules
A trip is dropped if any of these apply:
1. Duration under 1 minute or over 3 hours.
2. `trip_distance` is 0 or less, or over 100 miles.
3. `fare_amount` or `total_amount` is 0 or less (refunds and voids).
4. Average speed over 80 mph.
5. `payment_type` is 3 (no charge) or 4 (dispute).

Cash trips (`payment_type` 2) are kept, but their `tip_amount` is set to missing (NaN), because cash tips are not recorded.

## Acceptance criteria
- `test_clean_taxis.py` has a test for each rule and for the cash-tip rule, using small hand-made DataFrames.
- No row in the output breaks any rule.
- Running the script twice gives an identical output file.
- The input file is not modified.

## Performance budget
A 7-million-row month must run in under a minute and 4 GB of memory, so use vectorized pandas only.

## Out of scope
Zone names, and any analysis beyond cleaning.
```

### Quick quiz

What did the **vague** version do with the tips on cash trips?

 - A. Dropped the cash trips altogether
 - B. Set the tips to missing (NaN)
 - C. Left the tips as 0
 - D. Estimated the tips from the card trips

**C.** Lecture 1's bug survives the cleaning. Anyone who averages `tip_amount` over the "clean" data gets $2.64 instead of $3.66 for the trips where a tip was actually recorded.

### What the vague version decided for you

It did a careful job: it explored the data first, wrote tests, and printed what each rule dropped. But it had to make a lot of decisions to do that. From its script and its summary:

 - **Dropped:** exact duplicates; refunds *and* the original charge each one reverses; trips under 0 minutes or over 3 hours; zero distance; over 80 mph; tolls over $100
 - **Changed values:** removed what it decided was a double-counted surcharge from `extra` in about a third of all rows; turned 0 passengers and the "Unknown" zones into missing values
 - **Changed types:** `payment_type` and `ratecodeid` became text labels ("Cash", "JFK", ...), `store_and_fwd_flag` became True/False
 - **Kept:** no-charge and disputed trips, and cash tips as 0

Every one of these is defensible, and it told me about most of them. But they weren't *my* decisions, and some of them break other people's code.

### Quick quiz

Here's the exact line from the vague version's script that relabels the payment types:

In [ ]:
PAYMENT_TYPES = {1: "Credit card", 2: "Cash", 3: "No charge", 4: "Dispute",
                 5: "Unknown", 6: "Voided trip"}

cleaned = taxis.copy()
cleaned["payment_type"] = cleaned["payment_type"].map(PAYMENT_TYPES).astype("category")

A teammate's tip analysis, written against the original data, runs `cleaned[cleaned.payment_type == 1].tip_amount.mean()`. What do they get?

 - A. About 3.71, the same as before
 - B. An error
 - C. `nan`
 - D. 0

In [ ]:
cleaned[cleaned.payment_type == 1].tip_amount.mean()

In [ ]:
(cleaned.payment_type == 1).sum()

No row equals `1` any more, so the selection is empty and the mean of nothing is `nan`. No error and no warning, just like the silent failures we saw in Module A.

### What the spec version did

Let's compare the two sessions using their logs:

In [ ]:
import json

def summarise(path):
    log = [json.loads(line) for line in open(path)]
    result = log[-1]
    tool_calls = sum(block['type'] == 'tool_use'
                     for e in log if e['type'] == 'assistant'
                     for block in e['message']['content'])
    return {
        'turns': result['num_turns'],
        'tool calls': tool_calls,
        'permission denials': sum(e.get('subtype') == 'permission_denied' for e in log),
        'minutes': round(result['duration_ms'] / 60000, 1),
        'cost ($)': round(result['total_cost_usd'], 2),
    }

pd.DataFrame([summarise('../data/claude_code_session_vague.jsonl'),
              summarise('../data/claude_code_session_spec.jsonl')],
             index=['vague', 'spec'])

The spec version took longer and cost a little more, mostly because it actually checked the performance budget: it built a 7-million-row file, found the script used 3.6 GB at its peak, removed two unneeded copies of the table to get it down to 3.2 GB, and timed it at under 30 seconds.

The vague version just said it "should handle the 7M-row files".

It also checked every acceptance criterion, with tests at the edges of each rule (exactly 1 minute, exactly 80 mph...), and then listed what the spec *didn't* cover:

In [ ]:
from IPython.display import Markdown

spec_log = [json.loads(line) for line in open('../data/claude_code_session_spec.jsonl')]
summary = spec_log[-1]['result'] 
Markdown(summary[summary.index('**Decisions'):])

Those go straight back into the spec as decisions to make. That's the loop from the workflow table.

### The lessons

 - A vague request doesn't get you a worse agent, it gets you the agent's decisions instead of yours.
 - Writing the spec is where your domain knowledge goes in (like knowing that cash tips aren't recorded). The agent can't know that unless the data or you tell it.
 - The acceptance criteria and performance budget gave the agent something to *check*, so it did. Without them, "done" just means "it ran".
 - Both agents still made unrequested decisions; the spec version just made fewer and listed them.

## Writing a spec

There's a template in the course repository at [`project/spec_template.md`](../project/spec_template.md), with the taxi spec as an example. The sections are:

| Section | What goes in it |
|---|---|
| **Goal and user** | Who it's for and what they need it for |
| **Inputs** | The data: source, format, size now and later |
| **Outputs** | Exactly what gets produced |
| **Definitions and rules** | Every decision someone could reasonably make differently |
| **Acceptance criteria** | How you'll know it's done, as things a test can check |
| **Performance budget** | How much data, how fast, how much memory |
| **Out of scope** | What it should *not* do |
| **Open questions** | Decisions you haven't made yet |

The spec is part of Milestone 2. The "done" criteria in this Friday's proposal are a first draft of your acceptance criteria.

### Quick quiz

Which of these is a good acceptance criterion?

 - A. "The output data should be clean"
 - B. "Cash tips are missing (NaN), not 0"
 - C. "The code should be fast"
 - D. "Follow best practices"

**B**, because a test can check it. "Clean", "fast" and "best practices" just hand the decision back to the agent. "Fast" becomes a good criterion once it has numbers: "a 7-million-row month in under a minute".

### Using the agent to help

You don't have to write the spec alone. Some things that work well:

 - Ask the agent to explore the data and **list the decisions** a cleaning script would have to make, *without writing any code*. Then you make them.
 - Ask it to **interview you**: "Ask me questions until you could write a spec for this."
 - Ask for a **plan** before any code. In Claude Code, *plan mode* (Shift+Tab) lets it read files and propose a plan but not change anything until you approve. Other agents have similar modes, or you can just say "don't write any code yet".
 - Ask it to add anything it had to guess to the spec's **open questions**, as the spec version did.

*Tip: keep the spec in the repository (e.g. `SPEC.md`) and point to it from your `CLAUDE.md` or `AGENTS.md`, so every new session starts from the same decisions.*

### Summary so far

 - A vague request gets you the agent's decisions instead of yours
 - A spec records your decisions, including the ones only you can make
 - Acceptance criteria and a performance budget give the agent (and you) something to check

# Context management

Recall from Lecture 8 that the model has no memory: everything it knows about your task is in the **context**, which gets sent again on every step. And models get worse as the context gets longer ("context rot").

So what the agent can see is one of the main things you control.

## What goes into the context?

 - The harness's system prompt and tool descriptions (~13k tokens for Claude Code)
 - Your `CLAUDE.md` / `AGENTS.md`
 - Every message you send
 - Every file the agent reads and every command output it sees
 - Everything the agent writes, including the code it puts into files

Let's look at how the context grew during the spec session from last time:

In [ ]:
spec_log = load_session('spec')

calls = {}
for event in spec_log:
    if event['type'] == 'assistant':
        calls[event['message']['id']] = sum(event['message']['usage'].values())
context = pd.Series(calls.values(), index=range(1, len(calls) + 1))

ax = context.plot(marker='o', figsize=(6, 3))
ax.set_ylim(0, 40000)
ax.set_xlabel('model call')
ax.set_ylabel('tokens in context')

It started at ~19k tokens (before reading anything) and nearly doubled over 14 model calls. Where did the extra come from?

In [ ]:
def characters(log):
    sizes = {'tool results': 0, 'model (code and messages)': 0}
    for event in log:
        if event['type'] not in ('assistant', 'user'):
            continue
        for block in event['message']['content']:
            if block['type'] == 'tool_result':
                sizes['tool results'] += len(json.dumps(block['content']))
            elif block['type'] == 'tool_use':
                sizes['model (code and messages)'] += len(json.dumps(block['input']))
            elif block['type'] == 'text':
                sizes['model (code and messages)'] += len(block['text'])
    return pd.Series(sizes)

characters(spec_log)

In this session, about two thirds of what was added was the agent's *own* output: the scripts and tests it wrote go into the context too. The tool results were small because it was careful to print summaries (`describe()`, `value_counts()`) rather than whole tables.

That isn't always the case. A single `print(df)` of a big DataFrame, a long log file or a verbose error can add thousands of tokens in one step.

## What should stay out?

 - **Big outputs.** Ask for (and look for) summaries: `df.info()`, `df.head()`, `value_counts()`, not the whole table.
 - **Irrelevant files.** Point the agent at the files that matter rather than letting it read everything.
 - **Dead ends.** A long debugging detour leaves failed attempts in the context, and the agent can drift back to them.
 - **Secrets.** Anything in the context can be repeated, logged or leaked (more on Wednesday).

### Quick quiz

Which of these belongs in your project's `CLAUDE.md` / `AGENTS.md`?

 - A. The full data dictionary for your 40-column dataset
 - B. "Run the tests with `python -m pytest`; the data is in `data/raw/`"
 - C. Your notes from last week's debugging session
 - D. Your API key, so the agent can download the data

**B.** It's loaded into *every* session, so keep it short and always relevant: how to run things, where things are, conventions to follow.

The data dictionary (A) is useful, but put it in its own file and mention it in `CLAUDE.md`, so the agent reads it only when it needs it. Old notes (C) are mostly noise. And never D!

## Tools for managing context

| | What it does | When to use it |
|---|---|---|
| **Start fresh** (`/clear`, or a new session) | Empties the conversation | Each new task; when the agent is going in circles |
| **Compact** (`/compact`, or automatic when the context is nearly full) | Replaces the history with a summary | Long tasks where you want to keep going; note that details get lost |
| **Write things down** (spec, `CLAUDE.md`, a notes file, git commits) | Keeps decisions *outside* the context, so any session can read them | Always: this is the only memory that survives |
| **Sub-agents** | A separate agent with its own context does a side task and reports back a short summary | Searching a big codebase, reading lots of documentation |

Other harnesses have the same ideas under slightly different names.

### Quick quiz

You've spent an hour with an agent debugging a join. It now keeps reintroducing a bug you fixed 30 minutes ago. What's the best next step?

 - A. Keep correcting it
 - B. Tell it to be more careful
 - C. Write down what you've learned (in the spec or a notes file) and start a fresh session
 - D. Switch to a bigger model

**C.** The context is full of failed attempts, including the old bug. A fresh session that starts from a short, correct summary will usually do better than any amount of correcting.

# Verification

Remember the agent loop: the model asks for a tool, the harness runs it, and the result goes back into the context. That means an agent can only fix the mistakes it can *see*.

So the most useful thing you can give an agent is a way to check its own work, a **verifier**:

 - Running the code at all (errors and tracebacks)
 - **Tests**, especially ones that encode your decisions
 - Comparing against a known-good output (a "golden file")
 - Type checkers and linters
 - **Evals** for fuzzier outputs, like checking an LLM's answers against a set of examples

Writing the checks first and letting the agent iterate until they pass is sometimes called **evaluation-driven development**.

## Case study: the same task with and without tests

A small project with a slow function, `zone_summary` in `zones.py`, used by a weekly report that other people's dashboards read:

```python
def zone_summary(df):
    """Per pickup zone: number of trips, median fare, and mean tip % on card trips.

    Used by report.py to build the weekly report, so keep the output the same.
    """
    rows = []
    for zone in sorted(df["pulocationid"].unique()):
        trips = df[df["pulocationid"] == zone]
        card = trips[trips["payment_type"] == 1]
        if len(card) > 0:
            tip_pct = round((card["tip_amount"] / card["fare_amount"]).mean() * 100, 1)
        else:
            tip_pct = None
        rows.append({"zone": zone, "trips": len(trips),
                     "median_fare": trips["fare_amount"].median(), "tip_pct": tip_pct})
    return pd.DataFrame(rows)
```

I asked Claude Code (Opus 5.5) the same thing in two copies of the project:

> *"zone_summary in zones.py is far too slow on the full monthly files (about 7 million rows). Can you make it faster?"*

The only difference: one copy also had `test_zones.py`, and a line in `CLAUDE.md` saying to run it after any change.

### The tests

```python
def test_columns_and_index():
    out = zone_summary(trips([(5, 1, 10.0, 2.0), (3, 2, 8.0, 0.0)]))
    assert list(out.columns) == ["zone", "trips", "median_fare", "tip_pct"]
    assert list(out.index) == [0, 1]          # a plain 0..n-1 index, zone is a column
    assert list(out["zone"]) == [3, 5]        # sorted by zone

def test_zone_with_only_cash_trips_has_missing_tip(): ...

def test_tip_pct_uses_card_trips_and_is_rounded():
    out = zone_summary(trips([(1, 1, 10.0, 1.0), (1, 1, 30.0, 10.0), (1, 2, 20.0, 0.0)]))
    assert out.loc[0, "tip_pct"] == 21.7      # mean of 10% and 33.33%, rounded to 1 dp
    assert out.loc[0, "median_fare"] == 20.0  # median over all trips

def test_matches_saved_output_on_sample():
    expected = pd.read_csv("data/expected_zone_summary.csv")
    out = zone_summary(pd.read_csv("data/taxi_2019.csv"))
    pd.testing.assert_frame_equal(out, expected, check_dtype=False)
```

The last one is a **golden-file** test: I saved the old function's output before letting anyone touch it.

### Quick quiz

Which version ended up with a correct, faster `zone_summary`?

 - A. Only the one with tests
 - B. Only the one without tests
 - C. Both
 - D. Neither

**C**, both. These are both now standard `groupby` rewrites. So what difference did the tests make?

In [ ]:
def session_stats(log):
    result = log[-1]
    return {
        'turns': result['num_turns'],
        'permission denials': sum(e.get('subtype') == 'permission_denied' for e in log),
        'seconds': round(result['duration_ms'] / 1000),
        'cost ($)': round(result['total_cost_usd'], 2),
    }

pd.DataFrame([session_stats(load_session('notests')), session_stats(load_session('tests'))],
             index=['without tests', 'with tests'])

### Without tests, it built its own verifier

It checked its rewrite by loading the *old* function from git and comparing the outputs:

In [ ]:
for event in load_session('notests'):
    if event['type'] != 'assistant':
        continue
    for block in event['message']['content']:
        if block['type'] == 'tool_use' and 'HEAD:zones.py' in block['input'].get('command', ''):
            print(block['input']['command'])

That's a good habit, and worth asking for if an agent doesn't do it. But note:
 - "The same" meant *its* idea of the same: the sample file plus one edge case it thought of
 - It only worked because the old version was committed to git (so commit before you let an agent loose!)
 - It took longer and cost more, and ran into three permission prompts on the way

### With tests, "the same" meant what *I* decided

It read the tests, rewrote the function, ran the tests and stopped. It also told me about a case the tests don't cover:

In [ ]:
print(load_session('tests')[-1]['result'])

That missing-zone case is the next test to write. Tests encode your decisions in a form the agent can run, which is exactly what a spec's acceptance criteria should become.

### Quick quiz

Neither agent timed the *old* version. How much faster is the rewrite on 1 million rows?

 - A. About 4×
 - B. About 40×
 - C. About 400×
 - D. About 4,000×

In [ ]:
import time

def zone_summary_loop(df):
    rows = []
    for zone in sorted(df["pulocationid"].unique()):
        trips = df[df["pulocationid"] == zone]
        card = trips[trips["payment_type"] == 1]
        if len(card) > 0:
            tip_pct = round((card["tip_amount"] / card["fare_amount"]).mean() * 100, 1)
        else:
            tip_pct = None
        rows.append({"zone": zone, "trips": len(trips),
                     "median_fare": trips["fare_amount"].median(), "tip_pct": tip_pct})
    return pd.DataFrame(rows)

def zone_summary_groupby(df):  # the version with tests
    out = df.groupby("pulocationid")["fare_amount"].agg(trips="size", median_fare="median")
    card = df[df["payment_type"] == 1]
    tip_ratio = card["tip_amount"] / card["fare_amount"]
    out["tip_pct"] = (tip_ratio.groupby(card["pulocationid"]).mean() * 100).round(1).reindex(out.index)
    out.index.name = "zone"
    return out.reset_index()

In [ ]:
big = pd.concat([taxis] * 100, ignore_index=True)  # 1 million rows

for f in [zone_summary_loop, zone_summary_groupby]:
    start = time.perf_counter()
    f(big)
    print(f"{f.__name__}: {time.perf_counter() - start:.3f} s")

Only a few times faster! The loop was never "far too slow": with ~130 zones it only filters the table ~130 times, and each filter is fast vectorized code. I said it was slow, and both agents took my word for it.

A performance requirement needs a number and a baseline ("the 7M-row file takes 40 s; it needs to take under 5 s"), and someone has to measure. We'll come back to this properly in Module C.

## Where should the tests come from?

 - **From your spec.** Each acceptance criterion should become at least one test.
 - **From the current behaviour**, before you change anything: save the output as a golden file.
 - **From the agent**, if you review them. Tests the agent writes check *its* interpretation of the task, so read them as carefully as the code.

### Quick quiz

An agent opens a pull request saying "all tests pass". What should you check first?

 - A. Whether it changed any of the tests
 - B. The code style
 - C. The commit message
 - D. How long the tests take to run

**A.** An agent that's rewarded for passing tests will sometimes "fix" a failing test by weakening it, deleting it or special-casing the test input. It's well documented in coding agents, and easy to miss in a big diff. A test the agent may edit isn't really checking it.

## Summary

Today, we covered the following topics:
 - The agentic workflow: requirements → spec → plan → implement → verify
 - Vague requests get you the agent's decisions; a spec gets you yours
 - Writing a spec, and testable acceptance criteria
 - Context: what goes in, what to keep out, and when to start fresh
 - Verifiers: an agent can only fix what it can see, and tests encode your decisions
 - A performance claim needs a number and a baseline

On Wednesday, we'll look at security: what happens when something in the agent's context isn't on your side.